# EEGNet PyTorch Example

本示例展示如何使用 PyTorch 版本的 EEGNet 进行 EEG 信号分类。

## 内容
1. 环境设置
2. 数据加载与预处理
3. 模型创建
4. 模型训练
5. 模型评估
6. 结果可视化

## 1. 环境设置

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 导入 PyTorch 版本的 EEGNet
from EEGModels_PyTorch import EEGNet, DeepConvNet, ShallowConvNet

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

# 设置随机种子以保证可重复性
torch.manual_seed(42)
np.random.seed(42)

## 2. 数据加载与预处理

这里我们使用模拟数据作为示例。如果你有真实的 EEG 数据，请替换相应部分。

In [ ]:
# 模拟数据参数
n_classes = 4          # 分类类别数
n_channels = 64        # EEG 通道数
n_samples = 128        # 时间采样点数
n_train = 800          # 训练样本数
n_val = 200            # 验证样本数
n_test = 200           # 测试样本数

# 生成模拟数据
def generate_synthetic_eeg(n_trials, n_channels, n_samples, n_classes):
    """生成模拟 EEG 数据"""
    X = np.random.randn(n_trials, n_channels, n_samples, 1).astype(np.float32)
    y = np.random.randint(0, n_classes, n_trials)
    
    # 为每个类别添加一些特征模式
    for i in range(n_trials):
        class_idx = y[i]
        # 添加类别相关的信号模式
        freq = 10 + class_idx * 2  # 不同类别有不同的频率特征
        t = np.linspace(0, 1, n_samples)
        pattern = np.sin(2 * np.pi * freq * t) * 0.5
        X[i, :10, :, 0] += pattern[np.newaxis, :]  # 在前10个通道添加模式
    
    return X, y

# 生成训练、验证、测试数据
X_train, y_train = generate_synthetic_eeg(n_train, n_channels, n_samples, n_classes)
X_val, y_val = generate_synthetic_eeg(n_val, n_channels, n_samples, n_classes)
X_test, y_test = generate_synthetic_eeg(n_test, n_channels, n_samples, n_classes)

print(f'训练数据形状: {X_train.shape}')
print(f'验证数据形状: {X_val.shape}')
print(f'测试数据形状: {X_test.shape}')

In [ ]:
# 转换为 PyTorch 张量
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)
y_val_tensor = torch.LongTensor(y_val).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

# 创建 DataLoader
batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'训练批次数: {len(train_loader)}')
print(f'验证批次数: {len(val_loader)}')
print(f'测试批次数: {len(test_loader)}')

## 3. 模型创建

In [ ]:
# 创建 EEGNet 模型
model = EEGNet(
    nb_classes=n_classes,
    Chans=n_channels,
    Samples=n_samples,
    dropoutRate=0.5,
    kernLength=64,
    F1=8,
    D=2,
    F2=16,
    dropoutType='Dropout'
)

model = model.to(device)

# 打印模型结构
print(model)

# 计算参数数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')

In [ ]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 学习率调度器
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20
)

## 4. 模型训练

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        # 应用 max norm 约束
        model.apply_max_norm_constraints()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate_epoch(model, val_loader, criterion, device):
    """验证一个 epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# 训练循环
num_epochs = 100
best_val_acc = 0.0
best_model_path = 'best_model.pt'

# 记录训练历史
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print('开始训练...')
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    # 学习率调度
    scheduler.step(val_loss)
    
    # 记录历史
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
    
    # 打印进度
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print(f'\n训练完成! 最佳验证准确率: {best_val_acc:.4f}')

## 5. 模型评估

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load(best_model_path))
print(f'已加载最佳模型 (验证准确率: {best_val_acc:.4f})')

In [ ]:
# 在测试集上评估
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 计算准确率
test_acc = np.mean(all_preds == all_labels)
print(f'测试集准确率: {test_acc:.4f}')

In [ ]:
# 打印分类报告
class_names = [f'Class {i}' for i in range(n_classes)]
print('分类报告:')
print(classification_report(all_labels, all_preds, target_names=class_names))

## 6. 结果可视化

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot(train_losses, label='Train Loss', color='blue')
axes[0].plot(val_losses, label='Val Loss', color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# 准确率曲线
axes[1].plot(train_accs, label='Train Acc', color='blue')
axes[1].plot(val_accs, label='Val Acc', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 绘制混淆矩阵
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. 其他模型示例

除了 EEGNet，还可以使用 DeepConvNet 和 ShallowConvNet。

In [ ]:
# DeepConvNet 示例
print('DeepConvNet 模型结构:')
deep_model = DeepConvNet(nb_classes=n_classes, Chans=n_channels, Samples=n_samples)
deep_model = deep_model.to(device)
print(f'参数量: {sum(p.numel() for p in deep_model.parameters()):,}')

In [ ]:
# ShallowConvNet 示例
print('ShallowConvNet 模型结构:')
shallow_model = ShallowConvNet(nb_classes=n_classes, Chans=n_channels, Samples=n_samples)
shallow_model = shallow_model.to(device)
print(f'参数量: {sum(p.numel() for p in shallow_model.parameters()):,}')

## 8. 保存和加载模型

In [ ]:
# 保存完整模型
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accs': train_accs,
    'val_accs': val_accs,
    'best_val_acc': best_val_acc,
}, 'eegnet_checkpoint.pt')

print('模型已保存到 eegnet_checkpoint.pt')

In [ ]:
# 加载模型
checkpoint = torch.load('eegnet_checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print(f'模型已加载，最佳验证准确率: {checkpoint["best_val_acc"]:.4f}')

## 总结

本示例展示了如何使用 PyTorch 版本的 EEGNet 进行 EEG 信号分类，包括：

1. **数据准备**: 将 EEG 数据转换为 PyTorch 张量和 DataLoader
2. **模型创建**: 使用 EEGNet、DeepConvNet 或 ShallowConvNet
3. **训练**: 标准的 PyTorch 训练循环，包括 max norm 约束
4. **评估**: 测试集评估和混淆矩阵可视化
5. **模型保存**: 保存和加载训练好的模型

### 使用真实数据

要使用真实的 EEG 数据，只需替换数据加载部分：

```python
# 加载你的 EEG 数据
# X shape: (n_trials, n_channels, n_samples, 1)
# y shape: (n_trials,)
X = your_eeg_data  # numpy array
y = your_labels    # numpy array
```